# CADC: LIDC-IDRI nodule malignancy on Colab

Run the cells in order. Preprocessing and training both resume where they stopped,
so after a disconnect just reconnect and rerun from the top.
Use a T4/L4 for preprocessing (CPU and network bound) and an A100/L4 for training.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/CADC'
PATCHES = f'{DRIVE}/patches'
RUNS = f'{DRIVE}/runs'
REPO_URL = 'https://github.com/Dhir-learner/CAD-C.git'

In [ ]:
# Get or update the code.
import os
if os.path.isdir('/content/CADC/.git'):
    !git -C /content/CADC pull --ff-only
else:
    !git clone {REPO_URL} /content/CADC
%cd /content/CADC
!pip install -q -r requirements.txt

## 1. Preprocess (once, about 125 GB downloaded, a few GB of patches kept on Drive)

In [ ]:
# Try a few scans first; remove --limit for the full run (1,018 scans).
!python -m cadc.preprocess --out {PATCHES} --work /content/dicom --limit 5

## 2. Train

In [ ]:
# Train from a local copy; reading shards straight from Drive is slow.
!mkdir -p /content/patches && rsync -a --include='*.npz' --exclude='*' {PATCHES}/ /content/patches/
!ls /content/patches | wc -l

In [ ]:
RUN = f'{RUNS}/baseline'
for fold in range(5):
    !python -m cadc.train --data /content/patches --out {RUN} --fold {fold} --epochs 60 --batch-size 32

In [ ]:
# Pooled out-of-fold AUC across all folds.
import glob, pandas as pd
from sklearn.metrics import roc_auc_score
oof = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f'{RUN}/fold*/val_predictions.csv')))
print(f'{len(oof)} nodules, out-of-fold AUC = {roc_auc_score(oof.label, oof.prob):.4f}')